In [ ]:
from pathlib import Path
import numpy as np
import plotly.express as px

import openneuro
import mne
import mne_qt_browser

from mne_bids import (
    BIDSPath,
    find_matching_paths,
    get_entity_vals,
    make_report,
    print_dir_tree,
    read_raw_bids
)

import matplotlib.pyplot as plt
plt.switch_backend('Qt5Agg')
plt.ion()

In [ ]:
import sys
sys.path.insert(0, str(Path.cwd().parent / 'code'))
from pop_rejcont import rejcont
from preprocessing import preprocess
from psd_lof import find_bad_channels_psd_lof
from compute_psd_epochs import load_all

In [ ]:
dataset = "ds003969"
tasks = ['think1', 'think2','med1breath', 'med2']

# Create a BIDS root directory in the MNE dataset folder
root = Path('/home/jackhoyle')
bids_root = root / 'mne_data' / dataset
bids_root.mkdir(parents=True, exist_ok=True)

In [ ]:
by_task, freqs, ch_names, subjects = load_all(
    bids_root / 'outputs_v2/psd_epoch_mean', 
    key='psd_mean'
)

In [ ]:
import pandas as pd
subject_info = pd.read_csv(bids_root / 'participants.tsv', sep='\t').set_index('participant_id')

In [ ]:
import plotly.express as px

In [ ]:
import warnings

# mean spectrum per (channel, group, task), averaging over subjects and ignoring NaNs (bad channels)
ch_names = list(ch_names)
group_of = {s: subject_info.loc[f'sub-{s}', 'group'] for t in subjects for s in subjects[t]}

mean_spectra = {}
mean_se_spectra = {}
with warnings.catch_warnings():
    warnings.simplefilter('ignore', RuntimeWarning)  # all-NaN slices (channel bad for every subject)
    for task, arr in by_task.items():                 # arr: (n_subjects, n_channels, n_freqs)
        task_groups = np.array([group_of[s] for s in subjects[task]])
        for group in np.unique(task_groups):
            group_mean = 10**6 * np.nanmean(arr[task_groups == group], axis=0)   # (n_channels, n_freqs)
            group_mean_std = 10**6 * np.nanstd(arr[task_groups == group], axis=0)   # (n_channels, n_freqs)
            group_se = group_mean_std / np.sqrt(np.sum(~np.isnan(arr[task_groups == group]), axis=0))  # (n_channels, n_freqs)
            for ch_ix, ch in enumerate(ch_names):
                mean_spectra[(ch, group, task)] = group_mean[ch_ix]
                mean_se_spectra[(ch, group, task)] = group_se[ch_ix]

In [ ]:
import plotly.graph_objects as go
from plotly.colors import hex_to_rgb

ch, task = 'Oz', 'med1breath'
keys = {group: (c, group, t) for (c, group, t) in mean_spectra if c == ch and t == task}
df = pd.DataFrame({group: mean_spectra[k] for group, k in keys.items()}, index=freqs)
df.index.name = 'Frequency (Hz)'

fig = px.line(df, log_y=True, title=f'{ch} - {task}', labels={'value': 'PSD (V²/Hz)', 'variable': 'group'}, height=500)

# shaded mean ± SE band behind each line, in the same colour as the linex
for trace in fig.data:
    group = trace.name
    mean, se = mean_spectra[keys[group]], mean_se_spectra[keys[group]]
    lower = np.maximum(mean - se, mean * 1e-3)  # keep positive for the log axis
    upper = mean + se
    r, g, b = hex_to_rgb(trace.line.color)
    fig.add_trace(go.Scatter(
        x=np.concatenate([freqs, freqs[::-1]]),
        y=np.concatenate([upper, lower[::-1]]),
        fill='toself', fillcolor=f'rgba({r},{g},{b},0.3)', line=dict(width=0),
        hoverinfo='skip', showlegend=False, legendgroup=group,
    ))
fig

In [ ]:
[0,1,2][:99]

In [ ]:
group_colours = {
    group: colour
    for group, colour in zip(sorted(set(group_of.values())), px.colors.qualitative.Plotly)
}

def plot_individual_spectra(ch, task, scale=10**6, height=600):
    """One line per subject for a channel and task, coloured by group (for spotting outliers).
    Hover shows the subject id; click a group in the legend to toggle all of its subjects."""
    arr = by_task[task][:, ch_names.index(ch), :] * scale   # (n_subjects, n_freqs), NaN where channel was bad
    fig = go.Figure()
    seen_groups = set()
    for subj, spectrum in zip(subjects[task], arr):
        if np.isnan(spectrum).all():
            continue  # channel marked bad for this subject
        group = group_of[subj]
        fig.add_trace(go.Scatter(
            x=freqs, y=spectrum, mode='lines', name=group, legendgroup=group,
            showlegend=group not in seen_groups,
            line=dict(color=group_colours[group], width=1), opacity=0.6,
            hovertemplate=f'sub-{subj} ({group})<br>%{{x}} Hz: %{{y:.3g}}<extra></extra>',
        ))
        seen_groups.add(group)
    fig.update_layout(
        title=f'{ch} - {task} (individuals)', height=height,
        xaxis_title='Frequency (Hz)', yaxis_title='PSD (µV²/Hz)', yaxis_type='log',
        legend_title_text='group',
    )
    return fig

plot_individual_spectra('Oz', 'med2')